# Modelo Base — Inferência com TorchXRayVision
### DenseNet-121 pré-treinado — predição directa sem fine-tuning

In [ ]:
# Instalar dependências (só uma vez)
!pip install torchxrayvision scikit-image

In [ ]:
import torchxrayvision as xrv
import skimage.io
import torch
import torchvision


# ── 1. Carregar a imagem ──────────────────────────────────────────────────────
img = skimage.io.imread("00000011_006-acteletasis.png")  # muda para o teu ficheiro


# ── 2. Pré-processamento ──────────────────────────────────────────────────────
img = xrv.datasets.normalize(img, 255)  # converte para [-1024, 1024]

# CORRECÇÃO 1 — Conversão RGB → cinza documentada
# skimage.io.imread lê PNGs de 1 canal como arrays 2D (sem problema).
# Para imagens JPEG coloridas (3 canais), img.mean(2) faz a média simples
# dos canais R, G, B — aceitável para raio-X porque já são essencialmente cinza.
# Para imagens coloridas genuínas o correcto seria a fórmula de luminância:
#   img = 0.299 * img[:,:,0] + 0.587 * img[:,:,1] + 0.114 * img[:,:,2]
# Para o contexto de raio-X médico, img.mean(2) é suficiente.
if img.ndim == 3:
    img = img.mean(2)   # RGB → cinza (média simples — válida para raio-X)

img = img[None, ...]    # adiciona dimensão de canal → (1, H, W)

transform = torchvision.transforms.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(224)
])
img = transform(img)
img = torch.from_numpy(img)


# ── 3. Carregar modelo pré-treinado ──────────────────────────────────────────
# CORRECÇÃO 2 — Escolha explícita e documentada do peso do modelo
# Existem duas opções principais compatíveis com este projecto:
#
#   "densenet121-res224-mimic_ch"  → treinado no MIMIC-CXR (hospital de Boston, EUA)
#                                    labels CheXpert-style; inclui ~14 patologias
#
#   "densenet121-res224-chex"      → treinado directamente no CheXpert (Stanford, EUA)
#                                    ligeiramente diferente na distribuição de treino
#
# IMPORTANTE: usa o MESMO peso aqui e no notebook de transfer learning.
# O transfer learning vai adaptar este modelo base às imagens angolanas.
# Recomendação: manter "densenet121-res224-mimic_ch" (maior dataset de treino).
model = xrv.models.DenseNet(weights="densenet121-res224-mimic_ch")  # ← altera para "chex" se preferires
model.eval()  # modo de inferência (desativa dropout/batchnorm de treino)


# ── 4. Inferência ─────────────────────────────────────────────────────────────
with torch.no_grad():  # desativa gradientes (mais rápido, menos memória)
    outputs = model(img[None, ...])  # adiciona dimensão de batch → (1, 1, 224, 224)


# CORRECÇÃO 3 — Sigmoid obrigatório para obter probabilidades reais [0, 1]
# PROBLEMA ORIGINAL: o código usava `outputs` directamente como se fossem probabilidades.
# O modelo retorna logits (valores brutos). Sem sigmoid, um valor de -0.2 ou 1.3
# não representa percentagem — pode ser negativo ou superior a 100%.
# A função sigmoid converte qualquer valor real para o intervalo [0, 1]:
#   sigmoid(x) = 1 / (1 + e^(-x))
# Após esta correcção, cada valor representa a probabilidade real da doença.
probs = torch.sigmoid(outputs)  # ← CORRECÇÃO: converte logits → probabilidades [0, 1]


# ── 5. Resultados ─────────────────────────────────────────────────────────────
# CORRECÇÃO 3 (continuação) — usa `probs` em vez de `outputs`
resultados = dict(zip(model.pathologies, probs[0].detach().numpy()))

print("=" * 50)
print("       PREDIÇÕES DO MODELO (probabilidade real)")
print("=" * 50)
for doenca, prob in sorted(resultados.items(), key=lambda x: x[1], reverse=True):
    barra = "█" * int(prob * 20)
    print(f"{doenca:<35} {prob:.4f}  {barra}")
print("=" * 50)
print("\nNota: estes valores são do modelo base (dataset estrangeiro).")
print("Após fine-tuning com imagens angolanas, usa o notebook transfer_learning_multiclasse.ipynb")